# Pandas Basics

<img src="https://pandas.pydata.org/static/img/pandas.svg" width="300px" />

[Pandas](http://pandas.pydata.org/) is a an open source library providing Excel-like tables in Python.
It offers functionality for efficiently reading, writing, and processing data such as sorting, filtering, aggregating, and visualizing. Moreover, it provides tools for handling missing data and time series data.

<img src="https://media.geeksforgeeks.org/wp-content/cdn-uploads/creating_dataframe1.png" width="720px" />

:::{note}
Documentation for this package is available at https://pandas.pydata.org/docs/.
:::

## Package Imports

This will be our first experience with _importing_ a package.

Usually we import `pandas` with the _alias_ `pd`.

We might also need `numpy`, Python's main library for numerical computations.

In [ ]:
import pandas as pd
import numpy as np

## The `Series`

A Series represents a one-dimensional array of data. The main difference between a Series and numpy array is that a Series has an **index**. The index contains the labels that we use to access the data. It is actually quite similar to a Python dictionary, where each value is associated with a key.

There are many ways to [create a Series](https://pandas.pydata.org/pandas-docs/stable/dsintro.html#series), but the core constructor is [`pd.Series()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) which can process a dictionary to create a Series.

:::{note}
The data used below is from Wikipedia's [List of power stations in Germany](https://en.wikipedia.org/wiki/List_of_power_stations_in_Germany#Nuclear).
:::

In [ ]:
dictionary = {
    "Neckarwestheim": 1269,
    "Isar 2": 1365,
    "Emsland": 1290,
}
s = pd.Series(dictionary)
s

Arithmetic operations can be applied to `pd.Series`.

In [ ]:
s**0.5

We can access the underlying index object if we need to:

In [ ]:
s.index

We can get values back out using the index via the `.loc` attribute

In [ ]:
s.loc["Isar 2"]

We can pass a list or array to loc to get multiple rows back:

In [ ]:
s.loc[["Neckarwestheim", "Emsland"]]

## The `DataFrame`

There is a lot more to a `pandas.Series`, but they are limit to a single **column**. A more broadly useful Pandas data structure is the **DataFrame**. `pandas.DataFrame` is a collection of series that share the same index. It's a lot like a table in a spreadsheet.

The core constructor is `pd.DataFrame()`, which can be used like this using a dictionary of lists:

In [ ]:
data = {
    "capacity": [1269, 1365, 1290],  # MW
    "type": ["PWR", "PWR", "PWR"],
    "start_year": [1989, 1988, 1988],
    "end_year": [np.nan, np.nan, np.nan],
}
df = pd.DataFrame(data, index=["Neckarwestheim", "Isar 2", "Emsland"])
df

A wide range of statistical functions are available on both Series and DataFrames.

In [ ]:
df.min()

We can get a single column as a Series using python's getitem syntax on the DataFrame object.

In [ ]:
df["capacity"]

Indexing works very similar to series

In [ ]:
df.loc["Emsland"]

But we can also specify the column(s) and row(s) we want to access

In [ ]:
df.loc["Emsland", "start_year"]

Mathematical operations work as well, either on the whole DataFrame or on specific columns, the result of which can be assigned to a new column:

In [ ]:
df["reduced_capacity"] = df.capacity * 0.8
df

## Filtering Data

We can also filter a DataFrame using a boolean series obtained from a condition. This is very useful to build subsets of the DataFrame.

In [ ]:
df.capacity > 1300

In [ ]:
df[df.capacity > 1300]

## Modifying Values

In many cases, we want to modify values in a dataframe based on some rule. To modify values, we need to use `.loc` or `.iloc`

In [ ]:
df.loc["Isar 2", "capacity"] = 1366
df

## Time Series

Time indexes are great when handling time-dependent data.

Let's first read some time series data, using the `pd.read_csv()` function, which takes a local file path or a link to an online resource.

The **example data** contains hourly time series for Germany in 2015 for electricity demand, renewable capacity factors and day-ahead spot market prices.

In [ ]:
url = (
    "https://tubcloud.tu-berlin.de/s/pKttFadrbTKSJKF/download/time-series-lecture-2.csv"
)
ts = pd.read_csv(url, index_col=0, parse_dates=True)
ts.index[:5]

We can use Python's _slicing_ notation inside `.loc` to select a date range, and then use the built-in plotting feature of Pandas:

In [ ]:
ts.loc["2015-05-01":"2015-05-14", "solar"].plot()

A common operation is to change the resolution of a dataset by resampling in time, which Pandas exposes through the [resample](http://pandas.pydata.org/pandas-docs/stable/timeseries.html#resampling) function.

:::{note}
The resample periods are specified using pandas [offset index](http://pandas.pydata.org/pandas-docs/stable/timeseries.html#offset-aliases) syntax.
:::

In [ ]:
ts["solar"].resample("W").mean().plot()

## Grouping and Aggregation

Both `Series` and `DataFrame` objects have a `groupby` method, which allows you to **group and aggregate** the data based on the values of one or more columns.

It accepts a variety of arguments, but the simplest way to think about it is that you pass another series, whose unique values are used to split the original object into different groups.

Here's an example which retrieves the total generation capacity per country in Europe.

In [ ]:
fn = "https://raw.githubusercontent.com/PyPSA/powerplantmatching/master/powerplants.csv"

In [ ]:
df = pd.read_csv(fn, index_col=0)
df.iloc[:5, :10]

In [ ]:
grouped = df.groupby("Country").Capacity.sum()
grouped.sort_values(ascending=False).head(10)

Such **chaining** of multiple operations is very common with pandas.

Let's break apart this operation a bit. The workflow with `groupby` can be divided into three general steps:

1. **Split**: Partition the data into different groups based on some criterion.
1. **Apply**: Do some calculation (e.g. aggregation or transformation) within each group.
1. **Combine**: Put the results back together into a single object.

<img src="https://miro.medium.com/max/1840/1*JbF6nhrQsn4f-TaSF6IR9g.png" width="720px" />

Grouping is not only possible on a single columns, but also on multiple columns. For instance,
we might want to group the capacities by country **and** fuel type. To achieve this, we pass a list of functions to the `groupby` functions.

In [ ]:
capacities = df.groupby(["Country", "Fueltype"]).Capacity.sum().unstack().round(1)
capacities

## Exercises

### Power Plants Data

In this exercise, we will use the [powerplants.csv](https://raw.githubusercontent.com/PyPSA/powerplantmatching/master/powerplants.csv) dataset from the [powerplantmatching](https://github.com/PyPSA/powerplantmatching) project. This dataset contains information about various power plants, including their names, countries, fuel types, capacities, and more.

URL: `https://raw.githubusercontent.com/PyPSA/powerplantmatching/master/powerplants.csv`

**Task 1:** Load the dataset into a pandas DataFrame.

In [ ]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/PyPSA/powerplantmatching/master/powerplants.csv"
)
df = pd.read_csv(url, index_col=0)

**Task 2:** Provide a list of unique fuel types and technologies included in the dataset.

:::{note}
Look in the `pandas` documentation for functions that might be useful to solve these tasks.
:::

In [ ]:
df.Fueltype.unique()

In [ ]:
df.Technology.unique()

**Task 3:** Filter the dataset by power plants with the fuel type "Hard Coal".

In [ ]:
coal = df.loc[df.Fueltype == "Hard Coal"]
coal

**Task 4:** Identify the 5 largest coal power plants. In which countries are they located? When were they built?

In [ ]:
selection = coal.Capacity.nlargest(5).index
selection

In [ ]:
coal.loc[selection, ["Name", "Country", "Capacity", "DateIn"]]

### Wind and Solar Capacity Factors

In this exercise, we will work with a time series dataset containing hourly wind and solar capacity factors for the Philippines, taken from [model.energy](https://model.energy).

**Task 1:** Use `pd.read_csv` to load the dataset from the following URL into a pandas DataFrame. Ensure that the time stamps are treated as `pd.DatetimeIndex`.

In [ ]:
url = "https://model.energy/data/time-series-fe86a41f1102dcdc259428e80b379b42.csv"
df = pd.read_csv(url, index_col=0, parse_dates=True)

**Task 2:** Calculate the mean capacity factor for wind and solar over the entire time period.


In [ ]:
df.mean()

**Task 3:** Calculate and plot the weekly average capacity factors for wind and solar over the entire time period.

In [ ]:
df.resample("W").mean().plot()

**Task 6:** Go to [model.energy](https://model.energy) and retrieve the time series for another region of your choice. Recreate the analysis above and compare the results.

:::{note}
Look for "Download Comma-Separated-Variable (CSV) file of data" in Step 2.
:::